# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Turn the ranked queue into an editor playbook with limits and monitoring.


## 1. Ranked actions + reason codes

Use the reference pipeline queue (`outputs/refresh_queue.csv`) as the production-shaped
artifact; summarize top actions for the paper.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

qpath = ROOT / 'outputs' / 'refresh_queue.csv'
sample = ROOT / 'outputs' / 'refresh_queue_sample.csv'
path = qpath if qpath.exists() else sample
q = pd.read_csv(path)
print('Queue source:', path.relative_to(ROOT), 'rows:', len(q))
print('\nAction mix:')
print(q['suggested_action'].value_counts() if 'suggested_action' in q.columns else q.filter(like='action').iloc[:,0].value_counts())


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
Queue source: outputs\refresh_queue.csv rows: 30000

Action mix:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82
Name: count, dtype: int64


## 2. Intended use and limits

**Intended use:** Weekly editorial triage — open top-N high-confidence rows, verify on-page,
then decide refresh / CTR fix / expand.

**Limits:** Snapshot labels; no causal guarantee; starter ≠ full warehouse panel; low-confidence
rows are monitor-only.


In [2]:
print('Use: human-in-the-loop triage.')
print('Do not: auto-publish, claim Google prediction, skip on-page verification.')


Use: human-in-the-loop triage.
Do not: auto-publish, claim Google prediction, skip on-page verification.


## 3. Human review + the no-go list

**Always review:** legal pages, pricing, medical/YMYL-adjacent, pages with intentional freeze.

**No-go auto actions:** delete, unpublish, wholesale rewrite without editor sign-off.


In [3]:
nogo = [
    'auto-publish refreshed copy',
    'delete URL from score alone',
    'treat avg_position=0 as rank zero',
    'train featuring trend_direction/trend_pct',
]
print('No-go list:')
for x in nogo:
    print(' -', x)


No-go list:
 - auto-publish refreshed copy
 - delete URL from score alone
 - treat avg_position=0 as rank zero
 - train featuring trend_direction/trend_pct


## 4. Monitoring / retrain triggers

Retrain or freeze the queue when: Precision@50 on a fresh client-holdout drops toward baseline;
feature missingness spikes; content_type mix shifts sharply; warehouse panel replaces starter.


In [4]:
triggers = {
    'precision_at_50_floor': 0.40,
    'baseline_to_beat': float(json.loads((ROOT/'outputs'/'model_results.json').read_text())['baseline']['baseline_precision_at_50']),
    'retrain_if': 'holdout Precision@50 within 0.05 of baseline for 2 consecutive monthly refreshes',
}
print(json.dumps(triggers, indent=2))
(WORK_OUT/'playbook_triggers.json').write_text(json.dumps(triggers, indent=2))


{
  "precision_at_50_floor": 0.4,
  "baseline_to_beat": 0.24,
  "retrain_if": "holdout Precision@50 within 0.05 of baseline for 2 consecutive monthly refreshes"
}


162

## 5. Exports for the paper

Commit small JSON receipts + a markdown top-N table (not the full CSV).


In [5]:
# Build a public-safe top-10 markdown table for the paper
cols = [c for c in ['final_rank','refresh_priority_score','model_decline_probability',
                    'suggested_action','reason_codes','impressions_90d','sessions_90d','trend_direction'] if c in q.columns]
# flexible column names across sample vs full
if not cols:
    cols = list(q.columns)[:8]
top = q.head(10)[cols]
md_lines = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---']*len(cols)) + '|']
for _, r in top.iterrows():
    md_lines.append('| ' + ' | '.join(str(r[c])[:40] for c in cols) + ' |')
(WORK_OUT/'playbook_top10.md').write_text('\n'.join(md_lines))
print('\n'.join(md_lines[:6]), '...')
# metrics receipt
results = json.loads((ROOT/'outputs'/'model_results.json').read_text())
export = {
    'best_model': results['best_model']['name'],
    'precision_at_50': results['models']['random_forest']['precision_at_50'],
    'baseline_precision_at_50': results['baseline']['baseline_precision_at_50'],
    'base_rate': results['target_positive_rate'],
}
(WORK_OUT/'playbook_metrics.json').write_text(json.dumps(export, indent=2))
print('Wrote work/outputs/playbook_*.json/md')


| final_rank | suggested_action | impressions_90d | sessions_90d | trend_direction |
|---|---|---|---|---|
| 1 | refresh_and_review_ctr | 12834 | 66 | down |
| 2 | refresh_and_review_ctr | 2498 | 9 | down |
| 3 | refresh_and_review_ctr | 8064 | 23 | down |
| 4 | refresh_and_review_ctr | 13790 | 27 | down | ...


Wrote work/outputs/playbook_*.json/md


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
